# 🚀 Plateforme MLOps ITGate Group — Démonstration & Analyse du Modèle

Ce notebook présente l'évaluation approfondie du modèle de **Prévision du Chiffre d'Affaires (Time Series Multivarié)** d'ITGate Group, son intégration avec le **MLflow Model Registry**, l'analyse de l'importance des variables (**Feature Importance**) et la simulation de **Data Drift**.

## 1. Chargement et Exploration des Données Métier ITGate

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

# Configuration style graphique
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_PATH = "../data/raw/itgate_revenue_multivariate.csv"
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f"Dimensions du dataset : {df.shape[0]} mois enregistrés (2020-2023)")
df.head()

### Visualisation des Séries Temporelles (CA, Ingénieurs, Projets)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Graphique 1 : Chiffre d'Affaires
ax1.plot(df['date'], df['revenue'] / 1000, color='#0284c7', linewidth=2.5, marker='o', label='CA Réel (k TND)')
ax1.set_title("📈 Évolution du Chiffre d'Affaires Mensuel ITGate Group (2020 - 2023)", fontsize=13, fontweight='bold')
ax1.set_ylabel("CA (k TND)", fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.6)

# Graphique 2 : Drivers Métier
ax2.plot(df['date'], df['num_engineers'], color='#10b981', linewidth=2, label='Nombre d\'ingénieurs')
ax2.plot(df['date'], df['active_projects'], color='#f59e0b', linewidth=2, linestyle='--', label='Projets actifs')
ax2.set_title("👥 Évolution des Facteurs de Production (Ressources & Projets)", fontsize=13, fontweight='bold')
ax2.set_ylabel("Quantité", fontweight='bold')
ax2.set_xlabel("Date", fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 2. Ingénierie des Caractéristiques (Lag Features)

In [ ]:
lags = 3
for i in range(1, lags + 1):
    df[f'lag_{i}'] = df['revenue'].shift(i)

df_ml = df.dropna().reset_index(drop=True)
feature_cols = ['num_engineers', 'active_projects', 'avg_contract_value', 'lag_1', 'lag_2', 'lag_3']

X = df_ml[feature_cols]
y = df_ml['revenue']

test_size = 12
X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]
y_train, y_test = y.iloc[:-test_size], y.iloc[-test_size:]
dates_test = df_ml['date'].iloc[-test_size:]

print(f"Jeu d'entraînement : {X_train.shape[0]} mois | Jeu de test : {X_test.shape[0]} mois")
X.head()

## 3. Chargement du Modèle depuis MLflow Model Registry

In [ ]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
REGISTERED_MODEL_NAME = "ITGate_Revenue_Model"

try:
    model_uri = f"models:/{REGISTERED_MODEL_NAME}/Production"
    model = mlflow.sklearn.load_model(model_uri)
    print(f"✅ Modèle chargé avec succès depuis le MLflow Model Registry (Stage: Production)")
except Exception as e:
    print(f"⚠️ Chargement Registry échoué ({e}), fallback vers le dernier Run MLflow...")
    exp = mlflow.get_experiment_by_name("ITGate_Revenue_Forecast")
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=["start_time DESC"], max_results=1)
    run_id = runs.iloc[0].run_id
    model = mlflow.sklearn.load_model(f"runs:/{run_id}/random_forest_ts_model")
    print(f"✅ Modèle chargé depuis le Run MLflow {run_id}")

## 4. Évaluation & Graphique : CA Réel vs Prédictions

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("📊 Métriques sur l'année 2023 (Test Set) :")
print(f"   - R² Score : {r2:.4f}")
print(f"   - RMSE     : {rmse:.2f} TND")
print(f"   - MAE      : {mae:.2f} TND")

# Tracé graphique comparatif
plt.figure(figsize=(14, 6))
plt.plot(dates_test, y_test / 1000, 'o-', color='#0f172a', label='CA Réel (Test 2023)', linewidth=2.5)
plt.plot(dates_test, y_pred / 1000, 's--', color='#10b981', label='Prédictions Modèle RandomForest', linewidth=2.5)
plt.fill_between(dates_test, (y_pred - mae) / 1000, (y_pred + mae) / 1000, color='#10b981', alpha=0.15, label='Intervalle d\'incertitude (±MAE)')

plt.title("🎯 Comparaison CA Réel vs Prédictions Modèle ITGate (Test Set 2023)", fontsize=14, fontweight='bold')
plt.xlabel("Date", fontweight='bold')
plt.ylabel("Chiffre d'Affaires (k TND)", fontweight='bold')
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 5. Explicabilité : Importance des Variables (Feature Importance)

In [ ]:
if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    feat_names = ['Ingénieurs', 'Projets Actifs', 'Contrat Moyen', 'Revenu M-1', 'Revenu M-2', 'Revenu M-3']
    indices = np.argsort(importances)[::-1]

    plt.figure(figsize=(10, 5))
    bars = plt.barh(range(len(indices)), importances[indices], color='#0284c7', align='center')
    plt.yticks(range(len(indices)), [feat_names[i] for i in indices], fontweight='bold')
    plt.xlabel("Poids relatif dans la décision (%)", fontweight='bold')
    plt.title("🧠 Importance des Variables — Modèle ITGate Revenue", fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', linestyle='--', alpha=0.6)

    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f"{width*100:.1f} %", va='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

## 6. Simulation & Détection de Data Drift (Z-Score)

In [ ]:
import sys
sys.path.insert(0, '..')
from src.drift import detect_data_drift

# 1. Cas Normal (données d'inférence conformes)
normal_sample = pd.DataFrame([{
    'num_engineers': 45,
    'active_projects': 16,
    'avg_contract_value': 4800.0,
    'lag_1': 72500.0,
    'lag_2': 68000.0,
    'lag_3': 65400.0
}])
res_normal = detect_data_drift(normal_sample, file_path="../data/drift_baseline.json")
print("🟢 Test 1 — Inférence Normale :")
print(f"   Statut : {res_normal['status']} | Drift Score : {res_normal['drift_score']} (seuil : {res_normal['threshold']})")

# 2. Cas Dérive / Drift (ex: forte anomalie sur le nombre de projets et valeur contrats)
drift_sample = pd.DataFrame([{
    'num_engineers': 180,       # Valeur anormale (> 3x moyenne)
    'active_projects': 95,      # Valeur anormale
    'avg_contract_value': 18500.0,
    'lag_1': 250000.0,
    'lag_2': 230000.0,
    'lag_3': 210000.0
}])
res_drift = detect_data_drift(drift_sample, file_path="../data/drift_baseline.json")
print("\n🔴 Test 2 — Inférence avec Dérive Détectée :")
print(f"   Statut : {res_drift['status']} | Drift Score : {res_drift['drift_score']} (seuil : {res_drift['threshold']})")
print("   Détails des features en dérive :")
for feat, d in res_drift['details'].items():
    drift_flag = "⚠️ DÉVIATION" if d['drift'] else "✅ Normal"
    print(f"   - {feat:20s} : Z-score = {d['z_score']:5.2f} [{drift_flag}]")